In [ ]:
"""
TP-1: CleaningData — Applying OOP and Design Patterns
Exercises 1–5, using sample_data (2).csv.

Requirements:
    pip install pandas

Run:
    python tp1_cleaning_pipeline.py

By default, the script reads sample_data (2).csv from the same folder and
writes cleaned_data.csv to that folder.
"""

from __future__ import annotations

import logging
import time
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Callable, Any

import pandas as pd


# ---------------------------------------------------------------------------

## Exercise 3: Decorators for logging and timing

In [ ]:
# ---------------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)


def log_action(func: Callable) -> Callable:
    """Log when a method starts, finishes, or fails."""
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        logging.info("Starting %s", func.__qualname__)
        try:
            result = func(*args, **kwargs)
        except Exception:
            logging.exception("%s failed", func.__qualname__)
            raise
        logging.info("Finished %s", func.__qualname__)
        return result

    wrapper.__name__ = func.__name__
    wrapper.__doc__ = func.__doc__
    wrapper.__qualname__ = func.__qualname__
    return wrapper


def log_time(func: Callable) -> Callable:
    """Print the execution time of a method."""
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            logging.info("%s took %.6f seconds", func.__qualname__, elapsed)

    wrapper.__name__ = func.__name__
    wrapper.__doc__ = func.__doc__
    wrapper.__qualname__ = func.__qualname__
    return wrapper


# ---------------------------------------------------------------------------

## Exercise 1: CSVReader class

In [ ]:
# ---------------------------------------------------------------------------

class CSVReader:
    """Encapsulate loading and previewing a CSV file."""

    def __init__(self, file_path: str | Path):
        self._file_path = Path(file_path)
        self._data: pd.DataFrame | None = None

    @property
    def file_path(self) -> Path:
        """Read-only access to the input path."""
        return self._file_path

    @property
    def data(self) -> pd.DataFrame:
        """Return the loaded data, or raise an informative error."""
        if self._data is None:
            raise ValueError("CSV has not been loaded. Call read() first.")
        return self._data

    @log_action
    @log_time
    def read(self) -> pd.DataFrame:
        """Load the CSV file into a pandas DataFrame."""
        if not self._file_path.exists():
            raise FileNotFoundError(f"CSV file not found: {self._file_path}")
        self._data = pd.read_csv(self._file_path)
        logging.info("Loaded %d rows and %d columns", *self._data.shape)
        return self._data

    def preview(self, n: int = 5) -> None:
        """Print the first n rows."""
        if n < 0:
            raise ValueError("n must be zero or greater.")
        print(self.data.head(n))


# ---------------------------------------------------------------------------

## Exercise 2: Strategy Pattern for missing values

In [ ]:
# ---------------------------------------------------------------------------

class MissingValueStrategy(ABC):
    """Common interface for interchangeable missing-value strategies."""

    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a DataFrame with missing values handled."""


class DropMissing(MissingValueStrategy):
    """Remove rows containing one or more missing values."""

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    """Fill numeric missing values with each column's mean."""

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns
        for column in numeric_columns:
            if result[column].isna().any():
                mean_value = result[column].mean()
                if pd.notna(mean_value):
                    result[column] = result[column].fillna(mean_value)
        # For a usable complete dataset, fill non-numeric columns with mode.
        # If a column has no non-missing values, leave it unchanged.
        for column in result.select_dtypes(exclude="number").columns:
            if result[column].isna().any():
                modes = result[column].mode(dropna=True)
                if not modes.empty:
                    result[column] = result[column].fillna(modes.iloc[0])
        return result


class FillMode(MissingValueStrategy):
    """Fill missing values with the most frequent value in each column."""

    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.columns:
            if result[column].isna().any():
                modes = result[column].mode(dropna=True)
                if not modes.empty:
                    result[column] = result[column].fillna(modes.iloc[0])
        return result


class DataCleaner:
    """Apply the selected missing-value strategy."""

    def __init__(self, strategy: MissingValueStrategy):
        self._strategy = strategy

    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self._strategy.handle(df)


# ---------------------------------------------------------------------------

## Exercise 4: Factory Pattern for transformations

In [ ]:
# ---------------------------------------------------------------------------

class DataTransform(ABC):
    """Common interface for transformations."""

    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply a transformation and return the result."""


class NormalizeColumns(DataTransform):
    """Min-max scale numeric columns to the [0, 1] range, excluding ID fields."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        numeric_columns = result.select_dtypes(include="number").columns
        for column in numeric_columns:
            if column.lower() == "id" or column.lower().endswith("_id"):
                continue
            minimum = result[column].min()
            maximum = result[column].max()
            if pd.isna(minimum) or pd.isna(maximum):
                continue
            if maximum != minimum:
                result[column] = (result[column] - minimum) / (maximum - minimum)
            else:
                result[column] = 0.0
        return result


class RemoveDuplicates(DataTransform):
    """Remove duplicate rows and keep the first occurrence."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().copy()


class StandardizeText(DataTransform):
    """Trim extra spaces and apply title case to text columns."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include=["object", "string"]).columns:
            result[column] = (
                result[column]
                .astype("string")
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
                .str.title()
            )
        return result


class TransformFactory:
    """Create a transformation object from a string key."""

    _TRANSFORMS = {
        "normalize": NormalizeColumns,
        "normalize_columns": NormalizeColumns,
        "remove_duplicates": RemoveDuplicates,
        "standardize_text": StandardizeText,
    }

    @classmethod
    def create(cls, transform_name: str) -> DataTransform:
        key = transform_name.strip().lower()
        transform_class = cls._TRANSFORMS.get(key)
        if transform_class is None:
            choices = ", ".join(sorted(cls._TRANSFORMS))
            raise ValueError(
                f"Unknown transformation {transform_name!r}. Choose: {choices}"
            )
        return transform_class()


# ---------------------------------------------------------------------------

## Exercise 5: Template Method Pattern for the complete pipeline

In [ ]:
# ---------------------------------------------------------------------------

class DataPipeline(ABC):
    """Define the fixed load → clean → transform → save workflow."""

    def __init__(self, input_path: str | Path, output_path: str | Path):
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.df: pd.DataFrame | None = None

    def run(self) -> pd.DataFrame:
        """Template method: subclasses implement each workflow step."""
        self.load()
        self.clean()
        self.transform()
        self.save()
        if self.df is None:
            raise RuntimeError("Pipeline did not produce a DataFrame.")
        return self.df

    @abstractmethod
    def load(self) -> None:
        pass

    @abstractmethod
    def clean(self) -> None:
        pass

    @abstractmethod
    def transform(self) -> None:
        pass

    @abstractmethod
    def save(self) -> None:
        pass


class CSVDataPipeline(DataPipeline):
    """CSV pipeline using the reader, cleaner, and transformation factory."""

    def __init__(
        self,
        input_path: str | Path,
        output_path: str | Path,
        missing_strategy: MissingValueStrategy | None = None,
        transformations: list[str] | None = None,
    ):
        super().__init__(input_path, output_path)
        self.reader = CSVReader(input_path)
        self.cleaner = DataCleaner(missing_strategy or FillMean())
        self.transformations = transformations or [
            "remove_duplicates",
            "standardize_text",
            "normalize_columns",
        ]

    @log_action
    @log_time
    def load(self) -> None:
        self.df = self.reader.read()

    @log_action
    @log_time
    def clean(self) -> None:
        if self.df is None:
            raise RuntimeError("Load the data before cleaning.")
        self.df = self.cleaner.clean(self.df)

    @log_action
    @log_time
    def transform(self) -> None:
        if self.df is None:
            raise RuntimeError("Load the data before transforming.")
        for transform_name in self.transformations:
            transformation = TransformFactory.create(transform_name)
            self.df = transformation.apply(self.df)
            logging.info("Applied transformation: %s", transform_name)

    @log_action
    @log_time
    def save(self) -> None:
        if self.df is None:
            raise RuntimeError("There is no data to save.")
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        self.df.to_csv(self.output_path, index=False)
        logging.info("Saved cleaned data to %s", self.output_path)


# ---------------------------------------------------------------------------

## Run the assignment

In [ ]:
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    folder = Path(__file__).resolve().parent
    input_csv = folder / "sample_data (2).csv"
    output_csv = folder / "cleaned_data.csv"

    print("\n--- Exercise 1: Read and preview the CSV ---")
    reader = CSVReader(input_csv)
    reader.read()
    reader.preview(5)

    print("\n--- Missing values before cleaning ---")
    print(reader.data.isna().sum())

    print("\n--- Exercise 5: Run the complete cleaning pipeline ---")
    pipeline = CSVDataPipeline(
        input_path=input_csv,
        output_path=output_csv,
        missing_strategy=FillMean(),
        transformations=[
            "remove_duplicates",
            "standardize_text",
            "normalize_columns",
        ],
    )
    cleaned_df = pipeline.run()

    print("\n--- Cleaned data preview ---")
    print(cleaned_df.head())
    print("\n--- Missing values after cleaning ---")
    print(cleaned_df.isna().sum())
    print(f"\nRows: {len(cleaned_df)} | Columns: {len(cleaned_df.columns)}")
    print(f"Output saved to: {output_csv}")